## Load data + create input text + labels

In [100]:
import pandas as pd

df = pd.read_csv("quality_data.csv")

# Create input text
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

df["input_text"] = df["input_text"].str.lower()

# Create label
df["label"] = df["category"] + "_" + df["subcategory"]

df = df[["input_text", "label"]]
df.head()

,input_text,label
0,narration: emi auto debit sbi acc | mode: auto...,expense_loan
1,narration: salary credited via hdfc | mode: ne...,income_salary
2,narration: upi txn to zomato | mode: upi | typ...,expense_food
3,narration: paid electricity bill online | mode...,expense_bills
4,narration: mutual fund sip deducted | mode: au...,investment_mutual_funds


## Train Test setup

In [101]:
from sklearn.model_selection import train_test_split

X_train = df["input_text"]
y_train = df["label"]

X_test = df["input_text"]
y_test = df["label"]

## TF-IDF + Logistic Regression Model

In [102]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

tfidf_model.fit(X_train, y_train)

y_pred_tfidf = tfidf_model.predict(X_test)

In [103]:
print(y_test.value_counts())

label
income_salary              24
expense_bills              21
expense_loan               20
expense_food               20
expense_shopping           19
expense_transport          19
investment_fd_interest     16
expense_insurance          14
investment_stocks          14
investment_fd_booking       9
income_others               8
expense_health              7
investment_mutual_funds     4
expense_others              2
investment_others           1
Name: count, dtype: int64


## Prediction Confidence Calculation

In [104]:
import numpy as np

probs = tfidf_model.predict_proba(X_test)
confidence_scores = np.max(probs, axis=1)

## GLiNER Model Initialization

In [105]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_base")

labels = list(set(y_train))

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## TF-IDF Explanation Function

In [106]:
def explain_tfidf_pipeline(text, pipeline, top_n=3):
    import numpy as np
    
    vectorizer = pipeline.named_steps["tfidf"]
    model = pipeline.named_steps["clf"]
    
    vec = vectorizer.transform([text])
    feature_names = np.array(vectorizer.get_feature_names_out())
    scores = vec.toarray()[0]
    
    if scores.sum() == 0:
        return "No strong words → prediction based on weak similarity"
    
    # Top words from input
    top_indices = scores.argsort()[-top_n:][::-1]
    top_words = feature_names[top_indices]
    
    # Get predicted class
    pred_class = pipeline.predict([text])[0]
    
    return f"Words [{', '.join(top_words)}] influenced prediction → classified as '{pred_class}'"

## GLiNER Prediction Logic

In [107]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        return entities[0]["label"], entities[0]["score"]
    return "unknown", 0.0


y_pred_gliner = []
gliner_conf = []

for text in X_test:
    label, score = gliner_predict(text)
    y_pred_gliner.append(label)
    gliner_conf.append(score)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


## Classification Report + F2 Score

In [108]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_pred_tfidf, output_dict=True)

report_df = pd.DataFrame(report).transpose()

# Add F2 score manually
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)
print("\nClassification Report:")
print(report_df)


Classification Report:
                         precision  recall  f1-score  support  f2_score
expense_bills                0.955   1.000     0.977   21.000     0.991
expense_food                 0.952   1.000     0.976   20.000     0.990
expense_health               1.000   0.714     0.833    7.000     0.758
expense_insurance            1.000   1.000     1.000   14.000     1.000
expense_loan                 1.000   1.000     1.000   20.000     1.000
expense_others               0.000   0.000     0.000    2.000     0.000
expense_shopping             0.950   1.000     0.974   19.000     0.990
expense_transport            0.950   1.000     0.974   19.000     0.990
income_others                1.000   0.875     0.933    8.000     0.897
income_salary                0.889   1.000     0.941   24.000     0.976
investment_fd_booking        1.000   1.000     1.000    9.000     1.000
investment_fd_interest       1.000   0.938     0.968   16.000     0.949
investment_mutual_funds      1.000   1.0

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

## GLiNER Prediction Logic

In [109]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)
print("\nAccuracy:", round(accuracy, 3))


Accuracy: 0.965


In [110]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_insurance 1.0

Worst Performing Category:
expense_others 0.0


## Result DataFrame

In [111]:
result_df = pd.DataFrame({
    "input_text": X_test,
    "actual_label": y_test,
    "predicted_label": y_pred_tfidf,
    "confidence": confidence_scores
}).reset_index(drop=True)

## Model Explanation Column

In [112]:
result_df["model_comment"] = result_df["input_text"].apply(
    lambda x: explain_tfidf_pipeline(x, tfidf_model)
)

In [113]:
print(result_df.columns)

Index(['input_text', 'actual_label', 'predicted_label', 'confidence',
       'model_comment'],
      dtype='object')


## Category/Subcategory Split

In [114]:
# Actual split
result_df[["category", "subcategory"]] = result_df["actual_label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

## Confidence Formatting

In [115]:
result_df["confidence_percent"] = (result_df["confidence"] * 100).round(2).astype(str) + "%"

def confidence_level(c):
    if c >= 0.7:
        return "High"
    elif c >= 0.4:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

## Correct Flag

In [116]:
result_df["correct"] = result_df["actual_label"] == result_df["predicted_label"]

## Extract Fields

In [117]:
def extract_fields(text):
    parts = text.split("|")
    narration = parts[0].replace("narration:", "").strip()
    mode = parts[1].replace("mode:", "").strip()
    txn_type = parts[2].replace("type:", "").strip()
    return pd.Series([narration, mode, txn_type])

result_df[["narration", "mode", "type"]] = result_df["input_text"].apply(extract_fields)

## Output

In [118]:
final_df = result_df[[
    "narration",
    "mode",
    "type",
    "category",
    "subcategory",
    "predicted_category",
    "predicted_subcategory",
    "confidence",
    "confidence_percent",
    "confidence_level",
    "correct",
    "model_comment"
]]

final_df.head(20)

,narration,mode,type,category,subcategory,predicted_category,predicted_subcategory,confidence,confidence_percent,confidence_level,correct,model_comment
0,emi auto debit sbi acc,auto_debit,debit,expense,loan,expense,loan,0.437550,43.75%,Medium,True,"Words [sbi, acc, auto] influenced prediction →..."
1,salary credited via hdfc,neft,credit,income,salary,income,salary,0.590415,59.04%,Medium,True,"Words [hdfc, via, salary] influenced predictio..."
2,upi txn to zomato,upi,debit,expense,food,expense,food,0.335916,33.59%,Low,True,"Words [zomato, txn, to] influenced prediction ..."
3,paid electricity bill online,netbanking,debit,expense,bills,expense,bills,0.703791,70.38%,High,True,"Words [electricity, online, paid] influenced p..."
4,mutual fund sip deducted,auto_debit,debit,investment,mutual_funds,investment,mutual_funds,0.198886,19.89%,Low,True,"Words [sip, deducted, fund] influenced predict..."
5,got interest from fd,neft,credit,investment,fd_interest,investment,fd_interest,0.506932,50.69%,Medium,True,"Words [got, from, interest] influenced predict..."
6,amazon pay txn,upi,debit,expense,shopping,expense,shopping,0.372052,37.21%,Low,True,"Words [pay, txn, amazon] influenced prediction..."
7,credited bonus amount,neft,credit,income,salary,income,salary,0.488939,48.89%,Medium,True,"Words [bonus, amount, credited] influenced pre..."
8,fuel payment hp petrol,upi,debit,expense,transport,expense,transport,0.550964,55.1%,Medium,True,"Words [hp, fuel, petrol] influenced prediction..."
9,insurance premium paid,auto_debit,debit,expense,insurance,expense,insurance,0.652822,65.28%,Medium,True,"Words [premium, insurance, paid] influenced pr..."


## Classification Report + F2 Score

In [119]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred_tfidf, output_dict=True)
report_df = pd.DataFrame(report).transpose()

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [120]:
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)
report_df = report_df.round(3)
print("\nClassification Report:")
print(report_df)


Classification Report:
                         precision  recall  f1-score  support  f2_score
expense_bills                0.955   1.000     0.977   21.000     0.991
expense_food                 0.952   1.000     0.976   20.000     0.990
expense_health               1.000   0.714     0.833    7.000     0.758
expense_insurance            1.000   1.000     1.000   14.000     1.000
expense_loan                 1.000   1.000     1.000   20.000     1.000
expense_others               0.000   0.000     0.000    2.000     0.000
expense_shopping             0.950   1.000     0.974   19.000     0.990
expense_transport            0.950   1.000     0.974   19.000     0.990
income_others                1.000   0.875     0.933    8.000     0.897
income_salary                0.889   1.000     0.941   24.000     0.976
investment_fd_booking        1.000   1.000     1.000    9.000     1.000
investment_fd_interest       1.000   0.938     0.968   16.000     0.949
investment_mutual_funds      1.000   1.0

## Overall Metrics

In [124]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.965
Macro F1: 0.838
Weighted F1: 0.957


## Best & Worst Category

In [122]:
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()
print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])


Best Performing Category:
expense_insurance 1.0

Worst Performing Category:
expense_others 0.0


In [1]:
import joblib

# Save vectorizer
joblib.dump(vectorizer, "tfidf.pkl")

# Save classifier
joblib.dump(model, "clf.pkl")

NameError: name 'vectorizer' is not defined

In [ ]:
vectorizer = joblib.load("tfidf.pkl")
model = joblib.load("clf.pkl")